# Stage 4: ERAP2 Drug Design — DiffDock + RFdiffusion

Two approaches:
1. **DiffDock** — dock known aminopeptidase inhibitors against ERAP2 to validate the structure
2. **RFdiffusion** — generate de novo protein binders for ERAP2 active site

**Target:** ERAP2 (Q6P179) active site: H370, E371, H374, E393
**Structure:** AlphaFold DB AF-Q6P179-F1 (pLDDT 93.31)
**Experimental PDB:** 3SE6, 4E36, 5CU5 (for cross-validation)

In [ ]:
# Cell 1: Setup
import os, requests, torch
os.chdir("/content")

# Download ERAP2 structure
if not os.path.exists("erap2.pdb"):
    resp = requests.get("https://alphafold.ebi.ac.uk/files/AF-Q6P179-F1-model_v6.pdb")
    with open("erap2.pdb", "w") as f:
        f.write(resp.text)
    print(f"Downloaded ERAP2 AlphaFold structure")

# Also download experimental structure 3SE6 for comparison
if not os.path.exists("erap2_3SE6.pdb"):
    resp = requests.get("https://files.rcsb.org/download/3SE6.pdb")
    with open("erap2_3SE6.pdb", "w") as f:
        f.write(resp.text)
    print(f"Downloaded experimental structure 3SE6")

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print("Structures ready.")

---
## Part 1: DiffDock — Dock Known Inhibitors Against ERAP2

Validates that our ERAP2 structure produces realistic binding poses.
If known inhibitors dock correctly, the structure is good for novel design.

In [ ]:
# Cell 2: Install DiffDock
if not os.path.exists("DiffDock"):
    !git clone https://github.com/gcorso/DiffDock.git
    %cd DiffDock
    !pip install -q -e . 2>&1 | tail -5
    %cd /content
else:
    print("DiffDock already cloned")

# Check install
try:
    import diffdock
    print("DiffDock imported")
except ImportError:
    print("DiffDock installed (may use CLI instead of import)")

In [ ]:
# Cell 3: Define ligands to dock
# Known aminopeptidase inhibitors from DrugBank + literature
ligands = {
    "Bestatin": "CC(O)C(=O)NC(CC1=CC=CC=C1)C(O)=O",
    "Tosedostat": "CC(C)CC(NC(=O)C(CC1=CC=CC=C1)OC(=O)C(C)(C)C)B(O)O",
    "Leucinethiol": "CC(CC)CS",
    "Phosphoramidon": "CC(CC1=CC=CC=C1)C(=O)NC(CC(=O)O)C(=O)NC(CC2=CC=CC=C2)C(=O)O",
}

# Save ligands as SDF for DiffDock
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

os.makedirs("ligands", exist_ok=True)
valid_ligands = {}

for name, smiles in ligands.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"  {name}: invalid SMILES, skipping")
        continue
    
    # Add hydrogens and generate 3D
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    
    # Save as SDF
    sdf_path = f"ligands/{name}.sdf"
    writer = Chem.SDWriter(sdf_path)
    writer.write(mol)
    writer.close()
    
    mw = Descriptors.MolWt(Chem.MolFromSmiles(smiles))
    print(f"  {name}: MW={mw:.1f}, saved to {sdf_path}")
    valid_ligands[name] = sdf_path

# Also save SMILES as CSV for DiffDock's batch mode
with open("ligands/ligands.csv", "w") as f:
    f.write("complex_name,protein_path,ligand_description,protein_sequence\n")
    for name, smiles in ligands.items():
        if Chem.MolFromSmiles(smiles):
            f.write(f"{name},erap2.pdb,{smiles},\n")

print(f"\n{len(valid_ligands)} ligands ready for docking")

In [ ]:
# Cell 4: Run DiffDock
os.chdir("/content")

# DiffDock inference using CSV batch mode
!cd DiffDock && python -m inference \
    --protein_ligand_csv /content/ligands/ligands.csv \
    --out_dir /content/diffdock_results \
    --inference_steps 20 \
    --samples_per_complex 10 \
    --batch_size 4 \
    --no_final_step_noise \
    2>&1 | tail -30

# If the above fails, try the newer API:
# !cd DiffDock && python -m diffdock.inference \
#     --protein_ligand_csv /content/ligands/ligands.csv \
#     --out_dir /content/diffdock_results

In [ ]:
# Cell 5: Analyze DiffDock results
import glob

result_dirs = glob.glob("diffdock_results/*/")
print(f"DiffDock produced results for {len(result_dirs)} complexes")

for rdir in sorted(result_dirs):
    name = os.path.basename(rdir.rstrip("/"))
    sdfs = glob.glob(f"{rdir}/*.sdf")
    confs = glob.glob(f"{rdir}/*confidence*")
    
    print(f"\n{name}:")
    print(f"  Poses: {len(sdfs)}")
    
    # Read confidence scores if available
    if confs:
        with open(confs[0]) as f:
            scores = f.read().strip().split("\n")
        print(f"  Confidence scores: {scores[:5]}")
    
    # Read top pose
    if sdfs:
        top_sdf = sorted(sdfs)[0]
        mol = Chem.SDMolSupplier(top_sdf)[0]
        if mol:
            pos = mol.GetConformer().GetPositions()
            center = pos.mean(axis=0)
            print(f"  Top pose center: ({center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f})")

if not result_dirs:
    print("No results yet. Check cell 4 for errors.")
    !ls diffdock_results/ 2>/dev/null || echo "No results directory"

# Cell 7: Run RFdiffusion binder design against ERAP2
# Multiple lengths — let binding affinity decide, not cost
os.chdir("/content")
os.makedirs("rfdiffusion_results", exist_ok=True)

# Design binders at 4 different lengths (10 each = 40 total)
lengths = [
    ("short", "30-40"),    # Cheap peptide synthesis ~$500
    ("medium", "50-60"),   # Moderate cost ~$800
    ("long", "70-90"),     # Recombinant expression ~$1000
    ("large", "90-110"),   # Maximum binding surface ~$1500
]

for label, length_range in lengths:
    print(f"\n{'='*60}")
    print(f"Designing {label} binders ({length_range} residues)...")
    print(f"{'='*60}")
    
    !cd RFdiffusion && python scripts/run_inference.py \
        inference.output_prefix=/content/rfdiffusion_results/erap2_{label} \
        inference.input_pdb=/content/erap2.pdb \
        'contigmap.contigs=[A370-393/0 {length_range}]' \
        'ppi.hotspot_res=[A370,A371,A374,A392,A393]' \
        inference.num_designs=10 \
        inference.ckpt_override_path=models/Complex_beta_ckpt.pt \
        2>&1 | tail -5

import glob
total = len(glob.glob("rfdiffusion_results/*.pdb"))
print(f"\nTotal binder designs generated: {total}")

In [ ]:
# Cell 6: Install RFdiffusion
os.chdir("/content")

if not os.path.exists("RFdiffusion"):
    !git clone https://github.com/RosettaCommons/RFdiffusion.git
    %cd RFdiffusion
    
    # Install SE3-Transformers dependency
    !pip install -q e3nn
    !pip install -q --no-cache-dir dgl -f https://data.dgl.ai/wheels/torch-2.5/cu121/repo.html 2>&1 | tail -3
    !pip install -q -e . 2>&1 | tail -5
    
    # Download model weights
    !mkdir -p models
    if not os.path.exists("models/Complex_beta_ckpt.pt"):
        print("Downloading RFdiffusion model weights...")
        !wget -q --show-progress -O models/Complex_beta_ckpt.pt \
            http://files.ipd.uw.edu/pub/RFdiffusion/Complex_beta_ckpt.pt
        !wget -q --show-progress -O models/Base_ckpt.pt \
            http://files.ipd.uw.edu/pub/RFdiffusion/Base_ckpt.pt
    
    %cd /content
else:
    print("RFdiffusion already installed")

!ls -lh RFdiffusion/models/*.pt 2>/dev/null || echo "No model weights found"

In [ ]:
# Cell 7: Run RFdiffusion binder design against ERAP2
os.chdir("/content")
os.makedirs("rfdiffusion_results", exist_ok=True)

# Design 10 binders targeting ERAP2 active site
# Hotspot residues: H370, E371, H374, K392 (variant), E393
# contigmap: target chain A, binder length 70-100 residues
!cd RFdiffusion && python scripts/run_inference.py \
    inference.output_prefix=/content/rfdiffusion_results/erap2_binder \
    inference.input_pdb=/content/erap2.pdb \
    'contigmap.contigs=[A370-393/0 70-100]' \
    'ppi.hotspot_res=[A370,A371,A374,A392,A393]' \
    inference.num_designs=10 \
    inference.ckpt_override_path=models/Complex_beta_ckpt.pt \
    2>&1 | tail -20

In [ ]:
# Cell 8: Analyze RFdiffusion results
import glob

binder_pdbs = sorted(glob.glob("rfdiffusion_results/*.pdb"))
print(f"Generated {len(binder_pdbs)} binder designs")

for pdb_path in binder_pdbs:
    name = os.path.basename(pdb_path)
    with open(pdb_path) as f:
        lines = f.readlines()
    atom_lines = [l for l in lines if l.startswith("ATOM")]
    chains = set(l[21] for l in atom_lines)
    residues_per_chain = {}
    for l in atom_lines:
        ch = l[21]
        resnum = int(l[22:26].strip())
        if ch not in residues_per_chain:
            residues_per_chain[ch] = set()
        residues_per_chain[ch].add(resnum)
    
    chain_info = ", ".join(f"{ch}:{len(res)} res" for ch, res in sorted(residues_per_chain.items()))
    print(f"  {name}: {len(atom_lines)} atoms, chains: {chain_info}")

if not binder_pdbs:
    print("No binder PDBs found. Check cell 7 for errors.")
    !ls rfdiffusion_results/ 2>/dev/null

In [ ]:
# Cell 9: Visualize best binder
!pip install -q py3Dmol
import py3Dmol

if binder_pdbs:
    best = binder_pdbs[0]  # First design
    with open(best) as f:
        pdb_data = f.read()
    
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, "pdb")
    
    # Target (chain A) in gray
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "gray", "opacity": 0.7}})
    
    # Binder (chain B) in green
    view.setStyle({"chain": "B"}, {"cartoon": {"color": "green"}})
    
    # Hotspot residues in red
    for r in [370, 371, 374, 392, 393]:
        view.addStyle({"chain": "A", "resi": r}, {"stick": {"color": "red"}})
    
    view.zoomTo()
    print(f"Showing: {os.path.basename(best)}")
    print("Gray = ERAP2 target, Green = designed binder, Red sticks = active site")
    view.show()
else:
    print("No binder structures to visualize yet.")